Теперь обучим модель ruBert для классификации юридических текстов по категориям

# Импорт библиотек

In [1]:
! pip install transformers[torch] datasets evaluate optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 34.2 MB/s eta 0:00:00


In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

from datasets import Dataset, DatasetDict, load_dataset
import evaluate
import optuna
import random


In [18]:
import torch
import torch.nn as nn
import datasets

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from tqdm.auto import tqdm
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List
import string
import re
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, balanced_accuracy_score, roc_auc_score,
    precision_recall_fscore_support,
)
import joblib
from tqdm import tqdm
from sklearn.pipeline import make_pipeline
sns.set(palette='summer')

In [6]:
def set_seed(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1337)

# Загрузка данных

In [11]:
train_df=pd.read_csv('train_df.csv', sep=',')
val_df=pd.read_csv('val_df.csv', sep=',')
test_df=pd.read_csv('test_df.csv', sep=',')

Закондируем целевую переменную цифрами от 0 до 3

In [12]:
def labeling(category):
    if category=='Практика':
        return 0
    elif category=='Законодательство':
        return 1
    elif category=='Процесс':
        return 2
    elif category=='Другое':
        return 3
train_df['labels']=train_df['category'].apply(labeling)
train_df.drop('category', axis=1, inplace=True)

val_df['labels']=val_df['category'].apply(labeling)
val_df.drop('category', axis=1, inplace=True)

test_df['labels']=test_df['category'].apply(labeling)
test_df.drop('category', axis=1, inplace=True)

Слегка предобработаем текст для подачи в модель BERT:
1. приводим к нижнему регистру
2. убираем двойные пробелы
3. убираем частично пунктуацию

In [13]:
def preprocess(text):
    """
    Оптимальная предобработка для юридических текстов
    """
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?;:()\-]', '', text)

    return text.strip()

In [14]:
train_df['article']=train_df['article'].apply(preprocess)

val_df['article']=val_df['article'].apply(preprocess)

test_df['article']=test_df['article'].apply(preprocess)

In [15]:
train_df

,article,labels
0,утренний обзор за 2 августа. с соответствующим...,3
1,вс решит вопрос о выборе управляющего открытие...,2
2,"обзор сми за 1 июля. , сообщает коммерсант. в ...",3
3,обзор сми за 30 июля. . об этом пишет коммерса...,3
4,минпромторг предложил ввести промышленный сбор...,0
...,...,...
12687,в гд внесли законопроект об электронном кадров...,1
12688,субсидиарка за вывод активов и спор о сроках: ...,3
12689,важнейшие правовые темы в прессе обзор сми (1...,3
12690,путин ввел спецпорядок приема документов о мар...,1


Оборачиваем наши данные в Dataset

In [19]:
dataset_train=Dataset.from_pandas(train_df[['labels', 'article']])
dataset_val=Dataset.from_pandas(val_df[['labels', 'article']])
dataset_test=Dataset.from_pandas(test_df[['labels', 'article']])
dataset = DatasetDict({
    'train': dataset_train,
    'val': dataset_val,
    'test': dataset_test
})

In [20]:
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'article'],
        num_rows: 12692
    })
    val: Dataset({
        features: ['labels', 'article'],
        num_rows: 2720
    })
    test: Dataset({
        features: ['labels', 'article'],
        num_rows: 2720
    })
})

In [21]:
dataset['train'][0]

{'labels': 3,
 'article': 'утренний обзор за 2 августа. с соответствующими исками в арбитражный суд самарской области обратился первый зампрокурора региона игорь харитонов. он требует изъять из чужого незаконного владения два защитных сооружения гражданской обороны, которые находятся в корпусах молочного комбината эйч энд эн (ранее  данон россия). об этом говорится в материалах дел  а55-227092024 и  а55-227102024. в случае удовлетворения требований надзорного органа права собственности ответчика на эту недвижимость аннулируют. это следует из соответствующего законопроекта, разработанного минпромторгом. о документе пишут ведомости. еще одной новеллой станет то, что маркетплейсам запретят привлекать к работе самозанятых. представитель ведомства отметил, что пока точно не известно, когда инициативу внесут в думу и обсудят с профильными комитетами. дело в том, что подготовка комплексной редакцией законодательной инициативы все еще ведется вместе с депутатами и сенаторами. это тема, которой

# Обучение модели

Для обучения будем использовать ruBert-base модель, дообучим ее на наших данных и проверим на тестовых.
Создаем токенайзер и применяем его к нашим данным

In [23]:
base_model = 'ai-forever/ruBert-base'
tokenizer = AutoTokenizer.from_pretrained(base_model)
data_tokenized = dataset.map(lambda x: tokenizer(x['article'], truncation=True, max_length=256), batched=True, remove_columns=['article'])
data_tokenized

Map:   0%|          | 0/12692 [00:00<?, ? examples/s]

Map:   0%|          | 0/2720 [00:00<?, ? examples/s]

Map:   0%|          | 0/2720 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12692
    })
    val: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2720
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2720
    })
})

In [24]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

Создадим функцию для получения метрик качества предсказания

In [25]:
from sklearn.preprocessing import label_binarize
def compute_metrics(eval_pred):
    """Функция для подсчета метрик классификации"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)

    probabilities = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
    n_classes = logits.shape[1]
    labels_bin = label_binarize(labels, classes=range(n_classes))
    roc_auc = roc_auc_score(labels_bin, probabilities, multi_class='ovr', average='weighted')
    metrics = {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'roc_auc': roc_auc}
    return metrics

Инициируем модель

In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    base_model,
    num_labels=4,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)

pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

При обучении настроим следующие параметры:

eval_strategy = "epoch" — оценка качества модели после каждой эпохи;

save_strategy = "epoch" — сохранение чекпоинтов после каждой эпохи;

learning_rate = 4e-5 — скорость обучения;

per_device_train_batch_size = 16 — размер батча для обучения;

per_device_eval_batch_size = 32 — размер батча для валидации;

num_train_epochs = 5 — максимальное количество эпох обучения;

weight_decay = 0.05 — коэффициент L2-регуляризации для борьбы с переобучением;

metric_for_best_model = "accuracy" — метрика, по которой выбирается лучшая модель;

greater_is_better = True — чем выше accuracy, тем лучше;

logging_steps = 10 — логирование каждые 10 шагов;

fp16 = True — использование смешанной точности для ускорения обучения;

warmup_ratio = 0.06 — доля шагов для разогрева learning rate;

max_grad_norm = 1.0 — ограничение градиентов для стабильности;

lr_scheduler_type = "cosine" — косинусный планировщик скорости обучения.

In [27]:
args_es = TrainingArguments(
    output_dir="./rubert-early_stopping",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=4e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.05,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_dir='./logs',
    run_name="BERT-early_stopping",
    load_best_model_at_end=True,
    logging_steps=10,
    fp16=True,
    warmup_ratio=0.06,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# train_data_short = dataset['train'].select(range(5000))
# test_data_short = dataset['test'].select(range(700))
# def tokenize_function(examples):
#     return tokenizer(examples['article'], truncation=True, max_length=256)

# train_tokenized_short = train_data_short.map(tokenize_function, batched=True, remove_columns=['article', '__index_level_0__'])
# val_tokenized_short = test_data_short.map(tokenize_function, batched=True, remove_columns=['article', '__index_level_0__'])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Создаем Trainer, настраивает раннюю остановку, запускаем обучение

In [28]:
trainer = Trainer(
    model=model,
    args=args_es,
    train_dataset=data_tokenized['train'],
    eval_dataset=data_tokenized['val'],
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2,
        early_stopping_threshold=0.01)]
)

In [29]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Roc Auc
1,0.390045,0.406718,0.835662,0.842241,0.845331,0.841860,0.964901
2,0.247356,0.407243,0.854779,0.862768,0.863250,0.861623,0.968667
3,0.291512,0.400100,0.862500,0.874841,0.867555,0.870597,0.972683
4,0.158957,0.627511,0.847059,0.861026,0.856421,0.850063,0.970660


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3176, training_loss=0.3339976151509009, metrics={'train_runtime': 1005.8163, 'train_samples_per_second': 63.093, 'train_steps_per_second': 3.947, 'total_flos': 6678930961956864.0, 'train_loss': 0.3339976151509009, 'epoch': 4.0})

Результаты обучения модели:

Лучшая эпоха — 3:
Accuracy = 86.25%, F1 макро = 0.8706, ROC AUC = 0.9727

Динамика качества:
Модель стабильно улучшалась с 1 по 3 эпоху, достигнув пика на 3-й.

Признаки переобучения на 4-й эпохе:
Резкий рост валидационной ошибки (с 0.400 до 0.627) при падении accuracy до 84.7%.

Итог:
Оптимальной для использования является модель, обученная на 3 эпохах — она показывает лучший баланс между качеством и обобщающей способностью.

Сохраним модель на 3-ей эпохе

In [30]:
best_checkpoint = trainer.state.best_model_checkpoint
print(f"Лучший чекпоинт: {best_checkpoint}")

save_directory = "./rubert_base_model"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)


Лучший чекпоинт: ./rubert-early_stopping/checkpoint-2382


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./rubert_base_model/tokenizer_config.json',
 './rubert_base_model/tokenizer.json')

# Предсказания на тестовой выборке

Загрузим сохраненную модель и сделаем предсказания на тестовой выборке

In [34]:
model_path = "/content/rubert_base_model"

best_model = AutoModelForSequenceClassification.from_pretrained(model_path)
best_tokenizer = AutoTokenizer.from_pretrained(model_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model.to(device)
best_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(120138, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.2, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [36]:
predict_trainer = Trainer(
    model=best_model,
    processing_class=best_tokenizer,
    compute_metrics=compute_metrics)


In [37]:
predictions=predict_trainer.predict(data_tokenized['test'])

In [39]:
predictions.metrics

{'test_loss': 0.4117751121520996,
 'test_model_preparation_time': 0.0133,
 'test_accuracy': 0.8580882352941176,
 'test_precision_macro': 0.869377769070931,
 'test_recall_macro': 0.8648580071810024,
 'test_f1_macro': 0.8665391624250485,
 'test_roc_auc': 0.9706250352156378,
 'test_runtime': 49.45,
 'test_samples_per_second': 55.005,
 'test_steps_per_second': 6.876}

Результаты на тестовой выборке (не участвовала в обучении):

Accuracy: 85.81% — модель правильно классифицирует подавляющее большинство примеров;

F1 макро: 0.8665 — сбалансированное качество по всем классам;

ROC AUC: 0.9706 — исключительная способность разделять классы;

Precision макро: 0.8694 — высокая точность предсказаний;

Recall макро: 0.8649 — модель хорошо находит объекты каждого класса;

Время предсказания: 49.45 сек на весь тестовый набор (55 примеров/сек).

Вывод:
Модель показывает улучшение по всем метрикам посравнению с бейзлайном,  показывает стабильно высокие результаты на новых данных, не переобучена и готова к практическому использованию для классификации юридических текстов.
